# Chapter 4 — The Web Ontology Languages
### Notebook 5 · Agentic lab — axiomatisation under profile constraints

*Book reference: Extends §4.2 (OWL 2 features and profiles)*

Notebooks 1–4 taught you to *read and write* OWL 2 and to reason about its profiles. This lab builds an agent that does the writing — and, crucially, one that must respect the profile it was asked to target.

In [1]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


In [2]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch4_agentic as A
from oe_course import evaluation as ev, llm, mdp, optimize as opt
import oe_course, pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


**By the end of this notebook you can:**

1. Turn §4.2's profile restrictions into a **reward signal** an optimiser can act on.
2. Build a task whose score splits *faithfulness* from *profile compliance*, and see why that split matters.
3. Formalise axiomatisation as a **construction MDP** — where actions change the artefact — and contrast it with Chapter 1's evidence-gathering MDP.
4. Optimise the axiomatiser with GEPA and read which OWL 2 rules it learned.

> **Prerequisite.** This lab assumes Chapter 1 Notebook 5, which introduced tools, metrics, MDPs, GEPA and skills. Here we change the *task* and keep the discipline.

## 1. The task: requirement → axiom, inside a profile

A domain expert writes a requirement in English. The agent must emit an OWL axiom that is both a faithful reading **and** legal in the OWL 2 profile the project has committed to.

These two goals genuinely conflict, which is what makes the task worth studying. 'Giraffes eat nothing but leaves' is faithfully a *universal* restriction — and OWL 2 **EL has no universal restrictions at all**. An agent that only optimises faithfulness will hand you an ontology that falls out of your profile and loses you the polynomial-time reasoning you chose EL for.

In [3]:
print('Axiom shapes the agent may emit:', A.AXIOM_OPERATORS)
print()
print('Which profiles admit each shape (teaching-grade simplification):')
for op, profiles in A.PROFILE_TABLE.items():
    print(f'  {op:12s} {sorted(profiles)}')
print()
print('Note: EL lacks universal restrictions; QL lacks existential restrictions')
print('in the subclass position. Those two facts drive the whole exercise.')

Axiom shapes the agent may emit: ('subclassof', 'some', 'only', 'type')

Which profiles admit each shape (teaching-grade simplification):
  subclassof   ['EL', 'QL', 'RL']
  type         ['EL', 'QL', 'RL']
  some         ['EL', 'RL']
  only         ['RL']

Note: EL lacks universal restrictions; QL lacks existential restrictions
in the subclass position. Those two facts drive the whole exercise.


In [4]:
for r in A.REQUIREMENTS[:5]:
    print(f"{r['id']:22s} [{r['profile']}]  {r['text']}")
    print(f"{'':22s} gold: {r['axiom']}")

giraffe-isa            [EL]  Every giraffe is a herbivore.
                       gold: Giraffe SubClassOf Herbivore
lion-eats-some         [EL]  Every lion eats at least one herbivore.
                       gold: Lion SubClassOf (eats some Herbivore)
giraffe-eats-only      [RL]  Giraffes eat nothing but leaves.
                       gold: Giraffe SubClassOf (eats only Leaf)
impala-isa             [QL]  Every impala is a herbivore.
                       gold: Impala SubClassOf Herbivore
warthog-eats-some      [RL]  A warthog eats some plant.
                       gold: Warthog SubClassOf (eats some Plant)


### The axiom compiler and the reasoner are the tools

The agent emits *structure*, not OWL syntax. Deterministic code compiles that structure into real triples and a reasoner checks what it entails. Same division of labour as Chapter 1: **tools do what is mechanical, the model does what is interpretive** — which is also what keeps the output space small enough to grade.

In [5]:
ax = A.Axiom('Giraffe', 'some', 'Leaf', 'eats')
print('axiom      :', ax)
print('profiles OK:', sorted(A.profiles_allowing(ax)))
g = A.axiom_to_graph(ax)
print('\ncompiled to OWL:')
print(g.serialize(format='turtle'))

axiom      : Giraffe SubClassOf (eats some Leaf)
profiles OK: ['EL', 'RL']

compiled to OWL:
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

<http://example.org/ch4#Giraffe> a owl:Class ;
    rdfs:subClassOf [ a owl:Restriction ;
            owl:onProperty <http://example.org/ch4#eats> ;
            owl:someValuesFrom <http://example.org/ch4#Leaf> ] .

<http://example.org/ch4#Leaf> a owl:Class .

<http://example.org/ch4#eats> a owl:ObjectProperty .




In [6]:
chain = [A.Axiom('Giraffe', 'subclassof', 'Herbivore'),
         A.Axiom('Herbivore', 'subclassof', 'Animal')]
print('Giraffe SubClassOf Animal entailed?',
      A.entails_subclass(chain, 'Giraffe', 'Animal'))
print('...from only the first axiom?',
      A.entails_subclass(chain[:1], 'Giraffe', 'Animal'))

Giraffe SubClassOf Animal entailed? True


...from only the first axiom? False


## 2. The metric: faithfulness and compliance, scored separately

Half the mark for the right axiom, half for staying in profile. Collapsing these into one number would hide *which* half failed — and a metric that cannot say which half failed cannot drive GEPA (Ch. 1 §4.4).

In [7]:
train, dev = A.build_dataset('train'), A.build_dataset('dev')
print(f'train {len(train)}, dev {len(dev)}')

class Faithful:  # right reading, wrong profile
    axiom = json.dumps({'subject': 'Giraffe', 'operator': 'only',
                        'property': 'eats', 'filler': 'Leaf'})
example = [e for e in train if e.requirement_id == 'giraffe-eats-only'][0]
el_version = example.copy(profile='EL')   # same requirement, stricter profile
for label, ex in [('target RL', example), ('target EL', el_version)]:
    r = A.axiom_scorer(ex, Faithful())
    print(f'{label}: score={r.score}  violated={r.violated}')
    for n in r.notes: print('   ', n)

train 6, dev 4
target RL: score=1.0  violated=[]
    axiom_correct=1 profile_ok=1
target EL: score=0.5  violated=['respect-profile']
    Axiom uses 'only', which is outside OWL 2 EL (allowed there: ['some', 'subclassof', 'type']).
    axiom_correct=1 profile_ok=0


The *same* axiom scores 1.0 against RL and 0.5 against EL. That is the profile trade-off from §4.2 turned into a gradient the optimiser can follow.

## 3. Baseline, then GEPA

In [8]:
lm = llm.configure_dspy(A.AXIOM_RULEBOOK, A.axiom_responder)
baseline = A.AxiomProgram()
for ex in dev:
    pred = baseline(**ex.inputs())
    print(f'{ex.requirement_id:22s} -> {A.Axiom.parse(pred.axiom)}')
before = ev.evaluate_dataset(baseline, dev, A.axiom_scorer)
print('\nBEFORE:', before['mean_score'], before['violations'])

carnivore-eats-only    -> Carnivore SubClassOf (eats some Animal)
tree-isa               -> Tree Type Plant
branch-part-some       -> Branch SubClassOf (isPartOf only Tree)
rockdassie-isa         -> RockDassie Type Herbivore



BEFORE: 0.375 {'isa-is-subclassof': 2, 'only-for-universal': 1, 'some-for-existential': 1, 'respect-profile': 1}


In [9]:
metric = ev.make_gepa_metric(A.axiom_scorer, A.AXIOM_RULEBOOK)
reflect = llm.reflection_lm(A.AXIOM_RULEBOOK, A.axiom_responder)
tuned = opt.run_gepa(baseline, train, metric, valset=train,
                     max_metric_calls=40, reflection_lm=reflect)
result = opt.compare(A.AxiomProgram(), tuned, dev, A.axiom_scorer)
print(result.report())

2026/08/17 07:23:52 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 40 metric calls of the program. This amounts to 3.33 full evals on the train+val set.


2026/08/17 07:23:52 INFO dspy.teleprompt.gepa.gepa: Using 6 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.


GEPA Optimization:   0%|          | 0/40 [00:00<?, ?rollouts/s]

2026/08/17 07:23:52 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 6 (50.0%)


2026/08/17 07:23:52 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.5


...

2026/08/17 07:23:53 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 2 (100.0%)


2026/08/17 07:23:53 INFO dspy.teleprompt.gepa.gepa: Iteration 5: All subsample scores perfect. Skipping.


2026/08/17 07:23:53 INFO dspy.teleprompt.gepa.gepa: Iteration 5: Reflective mutation did not propose a new candidate


GEPA Optimization:  95%|█████████▌| 38/40 [00:00<00:00, 81.50rollouts/s]

mean score  0.375  ->  1.000   (delta +0.625)
violations  {'isa-is-subclassof': 2, 'only-for-universal': 1, 'some-for-existential': 1, 'respect-profile': 1}
        ->  {}

instruction diff:
--- instruction (before)
+++ instruction (after)
@@ -1 +1,5 @@
 You are an ontology engineer. Turn the requirement into an OWL axiom.
+- RULE some-for-existential: Translate 'at least one', 'some', or a bare plural object as an existential restriction (someValuesFrom), never a universal one.
+- RULE respect-profile: Check the requested OWL 2 profile before answering: EL has no universal restrictions and QL has no existential restrictions in the subclass position.
+- RULE only-for-universal: Translate 'only', 'nothing but', or 'exclusively' as a universal restriction (allValuesFrom).
+- RULE isa-is-subclassof: Translate 'every X is a Y' / 'an X is a Y' as a SubClassOf axiom between two classes; use rdf:type only for a named individual.


Read the diff: the optimiser recovered, from failure feedback alone, four rules that Chapter 4 spends pages establishing — the existential/universal distinction, is-a versus instance-of, and the profile restrictions. It did not *understand* OWL 2; it responded to a metric that punished each error specifically. That is worth being clear-eyed about: **the knowledge came from the metric**, and the metric came from you.

## 4. A construction MDP

Chapter 1's MDP had actions that only *bought information*. Here an action **changes the artefact**, so the formalisation differs:

| | Ch. 1 (evidence) | Ch. 4 (construction) |
|---|---|---|
| **S** | evidence gathered | axioms asserted so far |
| **A** | run a tool, or submit | assert a candidate axiom, or submit |
| **T** | deterministic | deterministic |
| **R** | −cost; score on submit | −cost; **entailment coverage − profile penalty** |

The reward now contains a *penalty term*, because a wrong action here does not merely waste money — it damages the artefact.

In [10]:
candidates = [
    A.Axiom('Giraffe', 'subclassof', 'Herbivore'),
    A.Axiom('Herbivore', 'subclassof', 'Animal'),
    A.Axiom('Giraffe', 'only', 'Leaf', 'eats'),     # illegal in EL
    A.Axiom('Giraffe', 'some', 'Leaf', 'eats'),     # legal in EL
]
M = A.AxiomConstructionMDP(candidates, required_entailments=[('Giraffe', 'Animal')],
                           profile='EL', step_cost=0.05, profile_penalty=0.5)
V, pi = mdp.value_iteration(M)
s0 = M.initial_state()
print(f'|S| = {len(M.states())}   V*(s0) = {V[s0]:.3f}')
plan = mdp.run_episode(M, mdp.greedy_policy(pi))
for a in plan.actions:
    print('  ', 'submit' if a == 'submit' else str(candidates[int(a.split(":")[1])]))

|S| = 32   V*(s0) = 0.900
   Giraffe SubClassOf Herbivore
   Herbivore SubClassOf Animal
   submit


The optimal plan asserts the two subsumptions that produce the required entailment **through a reasoner**, and declines both `eats` axioms — they cost money, add no required entailment, and one of them would have cost an extra 0.5 for leaving EL. Notice that the agent is rewarded for exploiting *entailment* rather than asserting `Giraffe SubClassOf Animal` directly: that is precisely the argument of Chapter 4 §4.3, expressed as a policy.

In [11]:
print('what happens if we drop the profile penalty:')
M2 = A.AxiomConstructionMDP(candidates, [('Giraffe', 'Animal')],
                            profile='EL', step_cost=0.05, profile_penalty=0.0)
V2, pi2 = mdp.value_iteration(M2)
print('  V* =', round(V2[M2.initial_state()], 3),
      '| plan:', mdp.run_episode(M2, mdp.greedy_policy(pi2)).actions)
print('\nThe plan is unchanged -- the illegal axiom was already not worth its\n'
      'step cost. A penalty only changes behaviour when the illegal action is\n'
      'otherwise attractive; see Exercise 4.2.')

what happens if we drop the profile penalty:


  V* = 0.9 | plan: ['assert:0', 'assert:1', 'submit']

The plan is unchanged -- the illegal axiom was already not worth its
step cost. A penalty only changes behaviour when the illegal action is
otherwise attractive; see Exercise 4.2.


### Exercise 4.1 — Try to make the profile penalty bite

Set up a case where an EL-illegal axiom is available, sweep `profile_penalty` from 0 to 1, and report the penalty at which the optimal policy stops using it. Then explain your result — it is probably not the one you expected.

> **Hint.** Build the MDP with only two candidates and sweep `profile_penalty`.

In [12]:
# YOUR CODE HERE


<details>
<summary>Solution 4.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [13]:
cands = [A.Axiom('Giraffe', 'only', 'Leaf', 'eats'),
         A.Axiom('Giraffe', 'subclassof', 'Herbivore')]
req = [('Giraffe', 'Herbivore')]
rows = []
for penalty in [0.0, 0.25, 0.5, 1.0]:
    Mp = A.AxiomConstructionMDP(cands, req, profile='EL',
                                step_cost=0.05, profile_penalty=penalty)
    Vp, pip = mdp.value_iteration(Mp)
    ep = mdp.run_episode(Mp, mdp.greedy_policy(pip))
    asserted = [str(cands[int(a.split(':')[1])]) for a in ep.actions if a != 'submit']
    illegal = [a for a in asserted if 'only' in a]
    rows.append({'penalty': penalty, 'V*': round(Vp[Mp.initial_state()], 3),
                 'n_asserted': len(asserted), 'breaks_EL': bool(illegal)})
print(pd.DataFrame(rows).to_string(index=False))
print('\nThe subsumption axiom alone satisfies the requirement legally, so the\n'
      'optimal policy never needs the EL-illegal axiom -- the penalty is not what\n'
      'protects the profile here; having a legal alternative is. That is the real\n'
      'lesson: reward shaping cannot rescue a candidate set with no legal option.')

 penalty   V*  n_asserted  breaks_EL
    0.00 0.95           1      False
    0.25 0.95           1      False
    0.50 0.95           1      False
    1.00 0.95           1      False

The subsumption axiom alone satisfies the requirement legally, so the
optimal policy never needs the EL-illegal axiom -- the penalty is not what
protects the profile here; having a legal alternative is. That is the real
lesson: reward shaping cannot rescue a candidate set with no legal option.


### Exercise 4.2 — Force the conflict

Now construct a case where **no** legal axiom satisfies the requirement, and decide what the agent should do. Argue for a reward design that makes the right choice optimal.

In [14]:
# YOUR CODE HERE


<details>
<summary>Solution 4.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [15]:
# Only an EL-illegal axiom can produce the required entailment.
cands = [A.Axiom('Giraffe', 'only', 'Leaf', 'eats')]
req = [('Giraffe', 'Leaf')]      # not entailed by a universal restriction
for penalty in [0.0, 0.5]:
    Mx = A.AxiomConstructionMDP(cands, req, profile='EL',
                                step_cost=0.05, profile_penalty=penalty)
    Vx, pix = mdp.value_iteration(Mx)
    ep = mdp.run_episode(Mx, mdp.greedy_policy(pix))
    print(f'penalty={penalty}: V*={Vx[Mx.initial_state()]:.2f} plan={ep.actions}')
print('\nAt every penalty -- including zero -- the agent submits an EMPTY ontology.\n'
      'The requirement is unsatisfiable from the available axioms, so asserting\n'
      'anything only costs. Silence is optimal here only because the reward has no\n'
      'way to express partial credit or escalation. The right engineering answer is\n'
      'to escalate (change the profile, or renegotiate the requirement). A reward\n'
      'function with only two options cannot express that, so a third action --\n'
      'ESCALATE, with a small negative reward -- belongs in the action set. Reward\n'
      'design is where you decide what your agent is allowed to do when it cannot win.')

penalty=0.0: V*=0.00 plan=['submit']


penalty=0.5: V*=0.00 plan=['submit']

At every penalty -- including zero -- the agent submits an EMPTY ontology.
The requirement is unsatisfiable from the available axioms, so asserting
anything only costs. Silence is optimal here only because the reward has no
way to express partial credit or escalation. The right engineering answer is
to escalate (change the profile, or renegotiate the requirement). A reward
function with only two options cannot express that, so a third action --
ESCALATE, with a small negative reward -- belongs in the action set. Reward
design is where you decide what your agent is allowed to do when it cannot win.


### Exercise 4.3 — Add a Chapter 4 rule the optimiser must discover

OWL 2 DL forbids non-simple (e.g. transitive) properties in cardinality restrictions — the rule behind Example 4.2. Add a `simple-property-only` rule and a requirement that punishes violating it, then show GEPA discovers it.

> **Hint.** Wrap `axiom_scorer`, halve the score when a transitive property appears under `only`, and add the rule id to `violated`.

In [16]:
# YOUR CODE HERE


<details>
<summary>Solution 4.3</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [17]:
from oe_course.llm import Rule, RuleBook
from oe_course.evaluation import ScoreReport

TRANSITIVE = {'isPartOf'}

def strict_scorer(gold, pred):
    report = A.axiom_scorer(gold, pred)
    ax = A.Axiom.parse(getattr(pred, 'axiom', None))
    if ax and ax.property in TRANSITIVE and ax.operator == 'only':
        report.score *= 0.5
        report.notes.append(
            f"'{ax.property}' is transitive; OWL 2 DL forbids non-simple properties "
            'in universal/cardinality restrictions (Example 4.2).')
        report.violated = list(dict.fromkeys(report.violated + ['simple-property-only']))
    return report

strict_rules = RuleBook(list(A.AXIOM_RULEBOOK) + [Rule(
    'simple-property-only',
    'Never use a transitive property (such as isPartOf) inside a universal or '
    'cardinality restriction; OWL 2 DL requires simple properties there.')])

lm2 = llm.configure_dspy(strict_rules, A.axiom_responder)
reflect2 = llm.reflection_lm(strict_rules, A.axiom_responder)
strict_metric = ev.make_gepa_metric(strict_scorer, strict_rules)
tuned2 = opt.run_gepa(A.AxiomProgram(), A.build_dataset('all'), strict_metric,
                      valset=A.build_dataset('all'), max_metric_calls=50,
                      reflection_lm=reflect2)
found = strict_rules.active_in(opt.instruction_of(tuned2))
print('rules discovered:', sorted(found))
print('\nWhether simple-property-only appears depends on whether the training set\n'
      'ever punished it. Check the violation histogram before concluding the\n'
      'optimiser failed -- an undiscovered rule usually means an unrepresented case.')
print('violations seen:',
      ev.evaluate_dataset(A.AxiomProgram(), A.build_dataset('all'), strict_scorer)['violations'])

2026/08/17 07:24:10 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 50 metric calls of the program. This amounts to 2.50 full evals on the train+val set.


2026/08/17 07:24:10 INFO dspy.teleprompt.gepa.gepa: Using 10 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.


GEPA Optimization:   0%|          | 0/50 [00:00<?, ?rollouts/s]

2026/08/17 07:24:10 INFO dspy.evaluate.evaluate: Average Metric: 4.5 / 10 (45.0%)


2026/08/17 07:24:10 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.45


...

2026/08/17 07:24:10 INFO dspy.teleprompt.gepa.gepa: Iteration 8: All subsample scores perfect. Skipping.


2026/08/17 07:24:10 INFO dspy.teleprompt.gepa.gepa: Iteration 8: Reflective mutation did not propose a new candidate


GEPA Optimization:  96%|█████████▌| 48/50 [00:00<00:00, 98.60rollouts/s] 


rules discovered: ['isa-is-subclassof', 'only-for-universal', 'respect-profile', 'simple-property-only', 'some-for-existential']

Whether simple-property-only appears depends on whether the training set
ever punished it. Check the violation histogram before concluding the
optimiser failed -- an undiscovered rule usually means an unrepresented case.


violations seen: {'isa-is-subclassof': 4, 'some-for-existential': 3, 'respect-profile': 2, 'only-for-universal': 2, 'simple-property-only': 1}


## Carrying this forward

Chapters 1 and 4 now share one scaffolding and differ only in the task:

| | Ch. 1 | Ch. 4 |
|---|---|---|
| task | assess an artefact | build an axiom |
| MDP | gather evidence | construct, under constraints |
| metric | level + defect F1 | faithfulness + profile compliance |
| what GEPA learns | reporting discipline | OWL 2 semantics and profile limits |

The pattern is the deliverable. Every remaining chapter plugs a new task into it — see `course/README.md` for the task, MDP and metric proposed for each.